# Creating an Audiobook from a PDF

### Steps
- Extract text from PDF file
- Clean the text
- Convert the text into speech
- Save the speech
- Play the speech

## 1. Extract text from PDF

### Install the library

In [10]:
#pip install PyPDF2

### Import the library

In [11]:
import PyPDF2

### Extract the text

In [12]:
import os

book_folder = "Book"
os.makedirs(book_folder, exist_ok=True)

print(f"Folder '{book_folder}' is ready.")
print(f"Please place your PDF file inside the '{book_folder}' folder.")


wpdf_files = []
while True:
    answer = input("Have you placed the PDF file in the 'Book' folder? (Yes/No): ").strip().lower()
    if answer == "yes":
        pdf_files = [f for f in os.listdir(book_folder) if f.lower().endswith(".pdf")]
        if pdf_files:
            break
        else:
            print("Unfortunately, no PDF file was found in the 'Book' folder. Please check and try again.")
    else:
        print("Please add the PDF file to the 'Book' folder, then type Yes.")


pdf_files = [f for f in os.listdir(book_folder) if f.lower().endswith(".pdf")]

if not pdf_files:
    raise FileNotFoundError(f"No PDF file found in the '{book_folder}' folder.")

pdf_path = os.path.join(book_folder, pdf_files[0])
print("Starting to extract text from your PDF file. This may take a few moments...")

extracted_text = ""

with open(pdf_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)
    num_pages = len(reader.pages)
    print(f"Number of pages: {num_pages}")

    for page_num in range(num_pages):
        page = reader.pages[page_num]
        page_text = page.extract_text()
        if page_text:
            extracted_text += page_text + "\n"

Folder 'Book' is ready.
Please place your PDF file inside the 'Book' folder.
Starting to extract text from your PDF file. This may take a few moments...
Number of pages: 9


### Clean the text

In [13]:
import re, unicodedata

ABBR = r"(?<!\bMr)(?<!\bMrs)(?<!\bMme)(?<!\bDr)(?<!\bSt)(?<!\bJr)"

def clean_text(text):
    
    text = re.sub(r"(?:/G[0-9A-F]{2}\s*)+", " ", text)
    text = unicodedata.normalize("NFKC", text)

    text = text.replace("\u2019", "'").replace("\u2018", "'")
    text = text.replace("\u201c", '"').replace("\u201d", '"')

    text = re.sub(r"\s*[\u2014\u2013]\s*", "; ", text)
    text = re.sub(r"\u2026|\.{3}", ",", text)

    text = re.sub(r"[ \t]*\n[ \t]*\n\s*", "\v", text)
    text = re.sub(r"[^\S\v]+", " ", text)
    text = text.replace("\v", ".\n\n")

    text = re.sub(r"[^\w\s.,;!?'\"-]", "", text)
    text = re.sub(r" *([.,;!?])", r"\1", text)
    text = re.sub(ABBR + r"([.!?]) +", r"\1\n", text)
    text = re.sub(r"\.\s*\.", ".", text)
    return text.strip()

cleaned_text = clean_text(extracted_text)

### Print the extracted text

In [14]:
print("Here is a preview of the cleaned text from your PDF file (first 2000 characters):")
print(cleaned_text[:2000])
print(f"\nTotal characters: {len(cleaned_text)}")

Here is a preview of the cleaned text from your PDF file (first 2000 characters):
.

THE GIFT OF THE MAGI.

BY O.
HENRY.

COPYRIGHT INFORMATION.

Short Story "The Gift of the Magi" Author O.
Henry William Sidney Porter, 1862; 1910 First published 1905.

The original story is in the public domain in the United States and in most, if not all, other countries as well.
Readers outside the United States should check their own countries' copyright laws to be certain they can legally download this ebook.
The Online Books Page has an FAQ which gives a summary of copyright durations for many other countries, as well as links to more official sources.

This PDF ebook was created by José Menéndez.
3.

NE dollar and eighty-seven cents.
That was all.
And sixty cents of it was in pennies.
Pennies saved one and two at a time by bulldozing the grocer and the vegetable man and the butcher until one's cheeks burned with the silent imputation of parsimony that such close dealing implied.
Three times Dell

## 2. Convert the Text into Speech

### Install the library

In [15]:
#pip install pyttsx3


In [16]:
#pip install pywin32

### Import the library

In [17]:
import pyttsx3

### Initialize a Speaker object

In [18]:
engine = pyttsx3.init()
voices = engine.getProperty("voices")

for i, v in enumerate(voices):
    print(f"[{i}] id: {v.id}")
    print(f"    name: {v.name}")
    print(f"    languages: {v.languages}")
    print(f"    gender: {v.gender}")
    print("-" * 40)

while True:
    choice = input(f"Choose a voice (0-{len(voices)-1}): ").strip()
    if choice.isdigit() and 0 <= int(choice) < len(voices):
        voice_index = int(choice)
        break
    print("Invalid choice, please try again.")

engine.setProperty("voice", voices[voice_index].id)
print(f"Your choice: {voices[voice_index].name}")

engine.setProperty("rate", 155)
engine.setProperty("volume", 1.0)

[0] id: HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Speech\Voices\Tokens\TTS_MS_EN-US_DAVID_11.0
    name: Microsoft David Desktop - English (United States)
    languages: ['en-US']
    gender: Male
----------------------------------------
[1] id: HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Speech\Voices\Tokens\TTS_MS_EN-US_ZIRA_11.0
    name: Microsoft Zira Desktop - English (United States)
    languages: ['en-US']
    gender: Female
----------------------------------------
[2] id: HKEY_LOCAL_MACHINE\SOFTWARE\Microsoft\Speech\Voices\Tokens\TTS_MS_RU-RU_IRINA_11.0
    name: Microsoft Irina Desktop - Russian
    languages: ['ru-RU']
    gender: Female
----------------------------------------
Your choice: Microsoft Zira Desktop - English (United States)


### Convert the text

In [19]:
engine.say(cleaned_text)

### Save the audio

In [20]:
output_path = "audiobook.mp3"

total_len = len(cleaned_text)
word_count = len(cleaned_text.split())
rate = engine.getProperty("rate")

estimated_minutes = word_count / rate
print(f"Approximate voice-over time: {estimated_minutes:.1f} min. (words: {word_count}, rate: {rate} words/min)")

def on_word(name, location, length):
    percent = min(100, int((location / total_len) * 100))
    print(f"\rProgress: {percent}%", end="")

engine.connect("started-word", on_word)

engine.save_to_file(cleaned_text, output_path)
engine.runAndWait()

print(f"\nAudiobook saved to {output_path}")

Approximate voice-over time: 11.1 min. (words: 2213, rate: 200 words/min)
Progress: 99%
Audiobook saved to audiobook.mp3
